# View the one-agent export

Select a local Python kernel and run the cells in order. Run the exporter first. This notebook reads the local CSV; it does not fetch transcripts or execute the recorded queries.

**Privacy:** outputs may contain confidential messages, queries and results. Clear all outputs before sharing or committing this notebook.

**Evidence limits:** QueryResult is already capped at 500 characters plus a truncation marker by the exporter. Blank fields mean no matching evidence was extracted. User-message and agent-response pairing is temporal, not proof of causality.

## 1. Load a compact overview

Run with the repository root as the working directory. If you changed OUTPUT_PATH or use a different working directory, set csv_path to your export's full path.

The overview intentionally shows only time, tool, status and duration. Use its **Row** number in the detail viewer below to read the question, query, result and response without a wide table. Requires pandas in your selected kernel; if missing, install it with `%pip install pandas`.

In [ ]:
from pathlib import Path
from html import escape
import json

import pandas as pd
from IPython.display import HTML, display

csv_path = Path.cwd() / "exports" / "one-agent.csv"
if not csv_path.is_file():
    raise FileNotFoundError(
        f"CSV not found: {csv_path}. Set csv_path to your export's full path."
    )

df = pd.read_csv(csv_path, dtype=str, keep_default_na=False)

df.index.name = "Row"

def show_overview(rows):
    table = rows[["ToolCallTime", "ToolName", "Status", "DurationMs"]].to_html(escape=True)
    display(HTML('<div style="max-height:420px;overflow:auto">' + table + '</div>'))

print(f"{len(df)} tool calls | {df['ConversationId'].nunique()} conversations | "
      f"{df['Status'].eq('Failed').sum()} failed calls")
print("Use the Row number in the detail viewer below.")
show_overview(df)

## 2. Find failed calls (optional)

Use the original Row number shown here in the next cell. No failed calls means an empty overview.

In [ ]:
show_overview(df.loc[df["Status"].eq("Failed")])

## 3. Read one tool call

Change **row_number** below and rerun this cell. Each field has its own wrapped, scrollable panel. All exported text is preserved; valid JSON results are indented. Expand **All exported fields** for IDs and remaining evidence. The viewer treats recorded content as text, not executable HTML or Markdown. The CSV must contain at least one row.

In [ ]:
row_number = 0
if not 0 <= row_number < len(df):
    raise IndexError(
        f"Row {row_number} is unavailable: the CSV contains {len(df)} rows."
    )
row = df.iloc[row_number]

def text_panel(label, value):
    text = str(value) if value != "" else "(Not recorded)"
    if label == "Query result / error snippet" and value:
        try:
            text = json.dumps(json.loads(value), indent=2, ensure_ascii=False)
        except json.JSONDecodeError:
            pass  # Truncated results and plain text remain exactly as exported.
    return (
        '<section style="margin:16px 0">'
        '<h4 style="margin-bottom:6px">' + escape(label) + '</h4>'
        '<pre style="white-space:pre-wrap;overflow-wrap:anywhere;max-height:360px;'
        'overflow:auto;padding:12px;border:1px solid #888;border-radius:6px">'
        + escape(text) + '</pre></section>'
    )

panels = [
    ("User question", "UserMessage"),
    ("Submitted query", "SqlQuery"),
    ("Query result / error snippet", "QueryResult"),
    ("Agent response", "AgentResponse"),
    ("Error code", "ErrorCode"),
    ("Error message", "ErrorMessage"),
]
heading = f"Row {row_number} | {row['Status']} | {row['ToolName']}"
html = '<article><h3>' + escape(heading) + '</h3>'
html += text_panel("Time (UTC)", row["ToolCallTime"])
html += text_panel("Conversation ID", row["ConversationId"])
html += ''.join(text_panel(label, row[column]) for label, column in panels)
html += '<details><summary>All exported fields</summary>'
html += ''.join(text_panel(column, value) for column, value in row.items())
html += '</details></article>'
display(HTML(html))